# Ordered Logistic Regression Results: FAIR² Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset (Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya) using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset schema and metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` identifiers.

In [ ]:
# List all available record sets with their @id and name for reference
print('Available Record Sets:')
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"@id: {rs['@id']}, name: {rs.get('name', '<no name>')}")
    
    # List fields/columns for each record set
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"    Field @id: {field['@id']}, name: {field.get('name', '<no name>')}")
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    for col in columns:
        print(f"    Column @id: {col['@id']}, name: {col.get('name', '<no name>')}")

## 3. Data Extraction
Load data from selected record set(s) using their `@id`s. We'll extract all record sets found above.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print('Record set @ids:', record_set_ids)

# Load records for each record set into pandas DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f'Loading records for: {record_set_id}')
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df

# Display columns of each DataFrame
for record_set_id, df in dataframes.items():
    print(f'Columns for {record_set_id}:', df.columns.tolist())
    if not df.empty:
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Perform basic analyses: filtering, normalization, and grouping using appropriate numeric and categorical fields by their `@id`. Update the variables below with real `@id`s from the previous section.

In [ ]:
# NOTE: You must set these to valid @id and column names from the outputs above.
# For illustration, we use the first available record set with data and try to auto-select numeric columns.
import numpy as np

selected_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        selected_record_set_id = rid
        break

if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    # Try to auto-select first numeric column
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

    if numeric_field:
        # Filtering example
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to select a grouping column (first object/categorical column)
        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (showing numeric means):")
            display(grouped_df.head())
        else:
            print("No categorical group field found in this DataFrame.")
    else:
        print("No numeric fields for EDA in selected record set.")
else:
    print("No record set with data found for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and, if possible, relations between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram and boxplot for the selected numeric field if available
if selected_record_set_id and numeric_field:
    plt.figure(figsize=(12,4))

    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Histogram of {numeric_field}")

    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f"Boxplot of {numeric_field}")

    plt.tight_layout()
    plt.show()

    # If group_field exists, plot means
    if group_field:
        mean_by_group = df.groupby(group_field)[numeric_field].mean().dropna()
        mean_by_group.plot(kind='bar', figsize=(8,4))
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset using `mlcroissant`, reviewed its structure, and performed basic exploratory analyses. For further domain-specific data analysis and advanced modeling, consider refining your field selections using `@id`s and exploring the domain documentation attached to the dataset.